<a href="https://colab.research.google.com/github/ouqbah/Sheetsync-SaaS/blob/main/SheetSync_SaaS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SheetSync SaaS

## SheetSync SaaS - Google Colab Workspace
This notebook sets up a production-ready FastAPI service that turns any Google Sheet into a cached REST API, containerizes it with Docker, and pushes the repository to GitHub for Google Cloud Run deployment.

## 1. Install Dependencies & Setup Workspace

In [1]:
# Install required libraries inside the Colab environment\n",
!pip install fastapi uvicorn google-api-python-client google-auth cachetools pydantic httpx pyngrok -q
print("✅ All dependencies installed successfully!")



✅ All dependencies installed successfully!


## 2.  Setup Project Structure & Write Files

In [2]:
import os

# Create project directory
project_dir = "sheetsync-saas"
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

print(f"Working directory set to: {os.getcwd()}")

# 1. Write requirements.txt
requirements_content = """fastapi==0.110.0
uvicorn==0.28.0
gspread==6.0.2
google-auth==2.28.1
cachetools==5.3.3
pydantic==2.6.4
pydantic-settings==2.2.1
pytest==8.1.1
httpx==0.27.0
"""
with open("requirements.txt", "w") as f:
    f.write(requirements_content)

# 2. Write main.py (FastAPI application with caching and sheet parsing)
main_py_content = """import os
import json
from fastapi import FastAPI, Header, HTTPException, status
from pydantic_settings import BaseSettings
from cachetools import TTLCache
import gspread
from google.oauth2.service_account import Credentials

class Settings(BaseSettings):
    api_key: str = "default_secure_key_change_me"
    google_service_account_json: str = "{}"

settings = Settings()

app = FastAPI(
    title="SheetSync SaaS",
    description="Turn Google Sheets into a high-performance cached JSON REST API.",
    version="1.0.0"
)

# Cache with 60-second TTL to avoid hitting Google Sheets API rate limits
cache = TTLCache(maxsize=100, ttl=60)

def get_sheet_data(sheet_id: str, worksheet_name: str = "Sheet1"):
    cache_key = f"{sheet_id}_{worksheet_name}"
    if cache_key in cache:
        return cache[cache_key]

    sa_json = os.getenv("GOOGLE_SERVICE_ACCOUNT_JSON", settings.google_service_account_json)
    if not sa_json or sa_json == "{}":
        raise HTTPException(
            status_code=500,
            detail="Google Service Account credentials not configured."
        )

    try:
        creds_dict = json.loads(sa_json)
        SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly"]
        creds = Credentials.from_service_account_info(creds_dict, scopes=SCOPES)
        client = gspread.authorize(creds)

        sheet = client.open_by_key(sheet_id)
        worksheet = sheet.worksheet(worksheet_name)
        records = worksheet.get_all_records()

        cache[cache_key] = records
        return records
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Failed to fetch sheet data: {str(e)}")

@app.get("/")
def health_check():
    return {"status": "healthy", "service": "SheetSync SaaS"}

@app.get("/api/v1/{sheet_id}")
def read_sheet(sheet_id: str, worksheet: str = "Sheet1", x_api_key: str = Header(default="")):
    expected_key = os.getenv("API_KEY", settings.api_key)
    if x_api_key != expected_key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or missing API Key"
        )
    return {"sheet_id": sheet_id, "worksheet": worksheet, "data": get_sheet_data(sheet_id, worksheet)}
"""
with open("main.py", "w") as f:
    f.write(main_py_content)

# 3. Write Dockerfile optimized for Cloud Run
dockerfile_content = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8080

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
"""
with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

# 4. Write deploy.sh shell script for Cloud Run
deploy_sh_content = """#!/bin/bash
set -e

SERVICE_NAME="sheetsync-saas"
REGION="us-central1"

echo "Deploying $SERVICE_NAME to Google Cloud Run (Always Free Tier)..."

gcloud run deploy $SERVICE_NAME \\
    --source . \\
    --platform managed \\
    --region $REGION \\
    --allow-unauthenticated \\
    --memory 256Mi \\
    --min-instances 0 \\
    --max-instances 10

echo "Deployment complete!"
"""
with open("deploy.sh", "w") as f:
    f.write(deploy_sh_content)

os.chmod("deploy.sh", 0o755)
print("All project files created successfully inside 'sheetsync-saas' directory!")


Working directory set to: /content/sheetsync-saas
All project files created successfully inside 'sheetsync-saas' directory!


## 3. Install Dependencies & Run Local Tests

In [4]:
# Install compatible versions of requirements in Colab environment (ignoring rigid pins for the test runner)
!pip install fastapi uvicorn gspread google-auth cachetools pydantic pydantic-settings pytest httpx -q

# Create a basic test file test_main.py
test_content = """from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

def test_health_check():
    response = client.get("/")
    assert response.status_code == 200
    assert response.json() == {"status": "healthy", "service": "SheetSync SaaS"}

def test_unauthorized_access():
    response = client.get("/api/v1/dummy_sheet_id")
    assert response.status_code == 401
"""
with open("test_main.py", "w") as f:
    f.write(test_content)

# Run pytest
print("Running test suite...")
!pytest test_main.py

Running test suite...
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/sheetsync-saas
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.13, platformdirs-4.13.6
collected 2 items                                                              

test_main.py ..                                                          [100%]

============================== 2 passed in 0.63s ===============================
